In [ ]:
import datetime as dt
import sys
import os
from pathlib import Path
import numpy as np
import random
import itertools
from numba import jit
from typing import Union
from numpy.typing import NDArray

# Add the workspace root to Python path so we can import from src
workspace_root = (
    Path(__file__).parent.parent.parent
    if "__file__" in globals()
    else Path.cwd().parent.parent
)
sys.path.insert(0, str(workspace_root))

import pandas as pd
import importlib
import src.utils.stochastic as stochastic
from tqdm.notebook import tqdm

# Model parameters
DISCOUNT_RATE = 0.01  # Discount rate
TRANSACTION_COST = 0.001  # Transaction cost
PVALUE_THRESHOLD = 0.01  # Only trade if we have 99.5% confidence
PERCENT_LOSS = 0.05
CASH_ALLOCATION = 1000

importlib.reload(stochastic)

In [ ]:
@jit(nopython=True)
def compute_trades(
    timestamp: np.ndarray,
    close_1: np.ndarray,
    close_2: np.ndarray,
    beta: np.ndarray,
    spread: np.ndarray,
    pvalue: np.ndarray,
    entry_level: np.ndarray,
    exit_level: np.ndarray,
    loss_level: np.ndarray,
    threshold_pvalue: float,
):
    # Setup the output data structures.
    trades = np.zeros_like(timestamp, dtype="int64")
    exit_reasons = np.zeros_like(
        timestamp, dtype="int64"
    )  # 0=none, 1=profit_target, 2=stop_loss
    n = trades.shape[0]

    # Setup cache.
    trade_open = False
    trade_beta: float = None
    trade_exit_level: float = None
    trade_loss_level: float = None

    for i in range(n):
        if trade_open:
            spread_i = close_1[i] - trade_beta * close_2[i]
            if spread_i > trade_exit_level:
                trades[i] = -1
                exit_reasons[i] = 1  # Profit target
                trade_open = False
                trade_beta = None
                trade_exit_level = None
                trade_loss_level = None
            elif spread_i < trade_loss_level:
                trades[i] = -1
                exit_reasons[i] = 2  # Stop loss
                trade_open = False
                trade_beta = None
                trade_exit_level = None
                trade_loss_level = None
        else:
            if (
                (spread[i] < entry_level[i])
                and (pvalue[i] < threshold_pvalue)
                and (spread[i] > loss_level[i])
            ):
                trades[i] = 1
                trade_open = True
                trade_beta = beta[i]
                trade_exit_level = exit_level[i]
                trade_loss_level = loss_level[i]

    return trades, exit_reasons


def calculate_pnl_with_costs(
    position_size: Union[float, int, NDArray[np.float64]],
    entry_price: Union[float, NDArray[np.float64]],
    exit_price: Union[float, NDArray[np.float64]],
    entry_time: Union[pd.Timestamp, NDArray],
    exit_time: Union[pd.Timestamp, NDArray],
    commission_rate: float = 0.001,
    borrow_rate_annual: float = 0.05,
    is_long: Union[bool, NDArray[np.bool_]] = None,
) -> Union[float, NDArray[np.float64]]:
    """
    Calculate PnL with transaction costs for long and short positions.

    Parameters
    ----------
    position_size : scalar or array
        Number of shares (can be positive, negative, or use is_long flag)
        If negative, treated as short position (unless is_long overrides)
    entry_price : scalar or array
        Entry price in $/share
    exit_price : scalar or array
        Exit price in $/share
    entry_time : pd.Timestamp or array of timestamps
        Entry timestamp for each trade
    exit_time : pd.Timestamp or array of timestamps
        Exit timestamp for each trade
    commission_rate : float, default 0.001
        Commission rate per side (e.g., 0.001 = 0.1%)
    borrow_rate_annual : float, default 0.05
        Annualized borrow cost for short positions (e.g., 0.05 = 5%)
    is_long : bool or array, optional
        Explicitly specify if position is long (True) or short (False)
        If None, inferred from sign of position_size

    Returns
    -------
    pnl : scalar or array
        Net PnL in $ (same shape as inputs)
    """
    # Convert to arrays
    position_size = np.asarray(position_size, dtype=np.float64)
    entry_price = np.asarray(entry_price, dtype=np.float64)
    exit_price = np.asarray(exit_price, dtype=np.float64)

    # Convert timestamps to numpy datetime64 if needed
    if isinstance(entry_time, pd.Timestamp):
        entry_time = np.array([entry_time], dtype="datetime64[ns]")
    elif isinstance(entry_time, (list, pd.DatetimeIndex)):
        entry_time = pd.to_datetime(entry_time).values
    else:
        entry_time = np.asarray(entry_time, dtype="datetime64[ns]")

    if isinstance(exit_time, pd.Timestamp):
        exit_time = np.array([exit_time], dtype="datetime64[ns]")
    elif isinstance(exit_time, (list, pd.DatetimeIndex)):
        exit_time = pd.to_datetime(exit_time).values
    else:
        exit_time = np.asarray(exit_time, dtype="datetime64[ns]")

    # Calculate holding period in days (including fractional days)
    time_delta = exit_time - entry_time
    holding_days = time_delta / np.timedelta64(1, "D")
    holding_days = holding_days.astype(np.float64)

    # Determine if positions are long or short
    if is_long is None:
        # Infer from sign of position_size
        is_long_arr = position_size >= 0
        abs_pos = np.abs(position_size)
    else:
        is_long_arr = np.asarray(is_long, dtype=bool)
        abs_pos = np.abs(position_size)

    # Calculate gross PnL
    pnl = np.where(
        is_long_arr,
        abs_pos * (exit_price - entry_price),  # Long PnL
        abs_pos * (entry_price - exit_price),  # Short PnL
    )

    # Commissions (both entry and exit)
    commissions = abs_pos * entry_price * commission_rate
    commissions += abs_pos * exit_price * commission_rate

    # Borrow cost (only for shorts)
    daily_rate = borrow_rate_annual / 365.0
    borrow_cost = np.where(
        ~is_long_arr,  # Only for shorts
        abs_pos * entry_price * daily_rate * holding_days,
        0.0,
    )

    net_pnl = pnl - commissions - borrow_cost

    # Return scalar if inputs were scalar
    return float(net_pnl) if net_pnl.ndim == 0 else net_pnl


def compute_pnl(
    df: pd.DataFrame,
    threshold_pvalue: float,
    cash_allocation: float,
    loss_percentage: float = 0.02,
):
    """
    Compute PnL using the beta from cointegration for proper hedging.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with trade signals and prices
    threshold_pvalue : float
        P-value threshold for entering trades
    cash_allocation : float
        Dollar amount to allocate to the LONG leg (e.g., $10,000)
    loss_percentage : float, default 0.02
        Maximum acceptable loss as percentage of TOTAL exposure (both legs)
    """
    df = df.copy()

    # Calculate position sizes
    long_shares = cash_allocation / df["close_1"]

    # Calculate total exposure for each pair (long leg + short leg)
    # Total exposure = long_dollar + short_dollar
    #                = cash_allocation + (beta * long_shares * close_2)
    #                = cash_allocation + (beta * cash_allocation)
    #                = cash_allocation * (1 + beta)
    total_exposure = cash_allocation * (1 + df["beta"])

    # Calculate max acceptable dollar loss based on TOTAL exposure
    max_dollar_loss = loss_percentage * total_exposure

    # How much can the spread move against us before we hit this loss?
    # When spread moves by $1 against us, we lose long_shares dollars
    adverse_spread_change = max_dollar_loss / long_shares

    # Loss level is BELOW entry (spread falling further = bad)
    df["loss_level"] = df["entry_level"] - adverse_spread_change

    # Compute the enter and exit trades with dynamic loss levels
    trades, exit_reasons = compute_trades(
        df["timestamp"].to_numpy(),
        df["close_1"].to_numpy(),
        df["close_2"].to_numpy(),
        df["beta"].to_numpy(),
        df["spread"].to_numpy(),
        df["pvalue"].to_numpy(),
        df["entry_level"].to_numpy(),
        df["exit_level"].to_numpy(),
        df["loss_level"].to_numpy(),
        threshold_pvalue,
    )

    # Only take data from the frame where we are either entering or exiting a trade.
    actual_trades = df[trades != 0].copy()
    actual_exit_reasons = exit_reasons[trades != 0]

    # If we have an odd number of trades, the last position is still open
    if len(actual_trades) % 2 != 0:
        last_row = df.iloc[-1].copy()
        last_row_df = pd.DataFrame([last_row])
        actual_trades = pd.concat([actual_trades, last_row_df], ignore_index=True)
        # Mark the forced exit as reason 3 (end of data)
        actual_exit_reasons = np.append(actual_exit_reasons, 3)

    # Re-shape the arrays
    ou_mu = actual_trades["mu"].to_numpy().reshape(-1, 2)
    ou_sigma = actual_trades["sigma"].to_numpy().reshape(-1, 2)
    ou_theta = actual_trades["theta"].to_numpy().reshape(-1, 2)
    close_1_prices = actual_trades["close_1"].to_numpy().reshape(-1, 2)
    close_2_prices = actual_trades["close_2"].to_numpy().reshape(-1, 2)
    times = actual_trades["timestamp"].to_numpy().reshape(-1, 2)

    # Reshape exit reasons (only exits have reasons, entries are 0)
    exit_reasons_reshaped = actual_exit_reasons.reshape(-1, 2)
    exit_reasons_codes = exit_reasons_reshaped[:, 1]  # Take the exit (second element)

    # Map exit reason codes to strings
    exit_reason_map = {0: "None", 1: "Profit Target", 2: "Stop Loss", 3: "End of Data"}
    exit_reasons_str = np.array([exit_reason_map[code] for code in exit_reasons_codes])

    # Get the beta values at entry points
    betas = actual_trades["beta"].to_numpy().reshape(-1, 2)
    entry_level = actual_trades["entry_level"].to_numpy().reshape(-1, 2)
    exit_level = actual_trades["exit_level"].to_numpy().reshape(-1, 2)
    entry_betas = betas[:, 0]
    entry_entry_levels = entry_level[:, 0]
    entry_exit_levels = exit_level[:, 0]

    # Get the spreads.
    spread_entry = close_1_prices[:, 0] - entry_betas * close_2_prices[:, 0]
    spread_exit = close_1_prices[:, 1] - entry_betas * close_2_prices[:, 1]

    # Position sizing
    long_position_sizes = cash_allocation / close_1_prices[:, 0]
    short_position_sizes = (cash_allocation / close_2_prices[:, 0]) / entry_betas

    # Compute PnL for each leg
    long_pnl = calculate_pnl_with_costs(
        long_position_sizes,
        close_1_prices[:, 0],
        close_1_prices[:, 1],
        times[:, 0],
        times[:, 1],
        is_long=True,
    )

    short_pnl = calculate_pnl_with_costs(
        short_position_sizes,
        close_2_prices[:, 0],
        close_2_prices[:, 1],
        times[:, 0],
        times[:, 1],
        is_long=False,
    )

    return pd.DataFrame(
        {
            "entry_time": pd.Series(times[:, 0], dtype="datetime64[ns]"),
            "exit_time": pd.Series(times[:, 1], dtype="datetime64[ns]"),
            "exit_reason": pd.Series(exit_reasons_str, dtype="str"),
            "ou_mu_entry": pd.Series(ou_mu[:, 0], dtype="float64"),
            "ou_mu_exit": pd.Series(ou_mu[:, 1], dtype="float64"),
            "ou_sigma_entry": pd.Series(ou_sigma[:, 0], dtype="float64"),
            "ou_sigma_exit": pd.Series(ou_sigma[:, 1], dtype="float64"),
            "ou_theta_entry": pd.Series(ou_theta[:, 0], dtype="float64"),
            "ou_theta_exit": pd.Series(ou_theta[:, 1], dtype="float64"),
            "short_entry_price": pd.Series(close_2_prices[:, 0], dtype="float64"),
            "short_exit_price": pd.Series(close_2_prices[:, 1], dtype="float64"),
            "short_pnl": pd.Series(short_pnl, dtype="float64"),
            "short_position_size": pd.Series(short_position_sizes, dtype="float64"),
            "long_entry_price": pd.Series(close_1_prices[:, 0], dtype="float64"),
            "long_exit_price": pd.Series(close_1_prices[:, 1], dtype="float64"),
            "long_pnl": pd.Series(long_pnl, dtype="float64"),
            "long_position_size": pd.Series(long_position_sizes, dtype="float64"),
            "hedge_ratio": pd.Series(entry_betas, dtype="float64"),
            "spread_entry": pd.Series(spread_entry, dtype="float64"),
            "spread_exit": pd.Series(spread_exit, dtype="float64"),
            "entry_level": pd.Series(entry_entry_levels, dtype="float64"),
            "exit_level": pd.Series(entry_exit_levels, dtype="float64"),
        }
    )

In [ ]:
window_days = 7
window = window_days * 24 * 60
test_days = 180
lookback_days = window_days + test_days
anchor_date = dt.date(2024, 12, 31)
end_time = dt.datetime.combine(anchor_date, dt.time.min)
start_time = end_time - dt.timedelta(days=lookback_days)
print(
    f"Running pairs trading over period {start_time.strftime('%Y-%m-%d')} to {end_time.strftime('%Y-%m-%d')}"
)

In [ ]:
ou_mu = 0.0001
ou_sigma = 0.001
ou_theta = 0.0001

exit_level = stochastic.OrnsteinUhlenbeck.get_optimal_exit_level(
    np.array([ou_mu]),
    np.array([ou_sigma]),
    np.array([ou_theta]),
    discount_rate=DISCOUNT_RATE,
    transaction_cost=TRANSACTION_COST,
    max_initial_shift=1000,
    max_iter=1000,
)[0]
entry_level = stochastic.OrnsteinUhlenbeck.get_optimal_entry_level(
    np.array([ou_mu]),
    np.array([ou_sigma]),
    np.array([ou_theta]),
    np.array([exit_level]),
    discount_rate=DISCOUNT_RATE,
    transaction_cost=TRANSACTION_COST,
    max_initial_shift=1000,
    max_iter=1000,
)[0]

gbm_params_1 = stochastic.GeometricBrownianMotionResult(mu=0.0, sigma=0.001)
gbm_1 = stochastic.GeometricBrownianMotion(gbm_params_1)
ou_params = stochastic.OrnsteinUhlenbeckResult(mu=ou_mu, sigma=ou_sigma, theta=ou_theta)
ou = stochastic.OrnsteinUhlenbeck(ou_params)
beta = 0.5
timestamps = pd.date_range(start=start_time, end=end_time, freq=dt.timedelta(minutes=1))
N = len(timestamps)
close_2 = gbm_1.simulate(N, 1, 100).flatten()
close_1 = beta * close_2 + ou.simulate(N, 1, ou_theta).flatten()
df = pd.DataFrame(
    {
        "timestamp": timestamps,  # Already a Series/array
        "close_1": close_1,  # Already flattened numpy array
        "close_2": close_2,  # Already flattened numpy array
        "spread": close_1 - beta * close_2,
        "beta": beta * np.ones_like(timestamps, dtype=float),
        "pvalue": 0.000001 * np.ones_like(timestamps, dtype=float),
        "mu": ou_mu * np.ones_like(timestamps, dtype=float),
        "sigma": ou_sigma * np.ones_like(timestamps, dtype=float),
        "theta": ou_theta * np.ones_like(timestamps, dtype=float),
        "entry_level": entry_level * np.ones_like(timestamps, dtype=float),
        "exit_level": exit_level * np.ones_like(timestamps, dtype=float),
    }
)

In [ ]:
pnl_df = compute_pnl(
    df,
    threshold_pvalue=PVALUE_THRESHOLD,
    cash_allocation=CASH_ALLOCATION,
    loss_percentage=PERCENT_LOSS,
)
pnl_df["total_pnl"] = pnl_df["long_pnl"] + pnl_df["short_pnl"]

In [ ]:
pnl_df[
    [
        "entry_time",
        "exit_time",
        "exit_reason",
        "total_pnl",
        "long_entry_price",
        "long_exit_price",
        "long_pnl",
        "short_entry_price",
        "short_exit_price",
        "short_pnl",
        "spread_entry",
        "spread_exit",
    ]
]

In [ ]:
import matplotlib.pyplot as plt

plt.plot(timestamps, df["spread"], label="Spread", color="black", linewidth=0.25)
plt.axhline(ou_theta, label="Theta", color="orange", linestyle="--")
plt.axhline(exit_level, label="Exit Level", color="red")
plt.axhline(entry_level, label="Entry Level", color="green")
plt.scatter(pnl_df["entry_time"], pnl_df["spread_entry"], marker="x")
plt.scatter(pnl_df["exit_time"], pnl_df["spread_exit"], marker="x")
plt.legend()

In [ ]:
print(f"Average PNL: {(pnl_df['long_pnl'] + pnl_df['short_pnl']).mean()}")
print(f"Total PNL: {(pnl_df['long_pnl'] + pnl_df['short_pnl']).sum()}")

In [ ]:
mask = trades != 0
exit_reason[mask]

In [ ]:
data_path = "/Users/glynfinck/Downloads/Kraken_OHLCVT"

In [ ]:
currency_files = []
for file in os.listdir(data_path):
    if file.endswith("USD_1.csv"):
        currency_files.append(file)

In [ ]:
def get_market_data(file_name: str, start_time: dt.datetime, end_time: dt.datetime):
    # Construct path.
    file_path = f"/Users/glynfinck/Downloads/Kraken_OHLCVT/{file_name}"

    try:
        # Read CSV without headers
        df = pd.read_csv(file_path, header=None)

        # Set meaningful column names for OHLCV data
        df.columns = [
            "timestamp",
            "open",
            "high",
            "low",
            "close",
            "volume",
            "trade_count",
        ]

        # Format the timestamp to be a datetime object
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s")
        df["file_name"] = file_name
        df["open"] = df["open"].astype(float)
        df["high"] = df["high"].astype(float)
        df["low"] = df["low"].astype(float)
        df["close"] = df["close"].astype(float)
        df["volume"] = df["volume"].astype(float)
        df["trade_count"] = df["trade_count"].astype(int)
        df.drop(columns=["trade_count"], inplace=True)
        df = df.sort_values("timestamp")

        df = df.loc[(df["timestamp"] >= start_time) & (df["timestamp"] <= end_time)]

        return df[["timestamp", "file_name", "close", "volume"]].set_index("timestamp")
    except:
        return pd.DataFrame(
            {
                "timestamp": pd.Series([], dtype="datetime64[ns]"),
                "file_name": pd.Series([], dtype="str"),
                "close": pd.Series([], dtype="float64"),
                "volume": pd.Series([], dtype="float64"),
            }
        ).set_index("timestamp")

In [ ]:
avg_daily_volume_usds = []
for file_name in tqdm(currency_files):
    df = get_market_data(file_name, start_time=start_time, end_time=end_time)
    avg_daily_volumes = df["volume"].rolling(dt.timedelta(days=1)).sum()
    avg_daily_price = df["close"].rolling(dt.timedelta(days=1)).mean()
    avg_daily_volume_usd = (avg_daily_volumes * avg_daily_price).mean()
    avg_daily_volume_usds.append(avg_daily_volume_usd)

In [ ]:
avg_daily_volume_df = (
    pd.DataFrame(
        {"file_name": currency_files, "avg_daily_volume_usd": avg_daily_volume_usds}
    )
    .dropna()
    .sort_values("avg_daily_volume_usd", ascending=False)
)
avg_daily_volume_df

In [ ]:
avg_daily_volume_df = avg_daily_volume_df.loc[
    ~avg_daily_volume_df["file_name"].str.startswith("USD")
]
avg_daily_volume_df = avg_daily_volume_df.loc[
    ~avg_daily_volume_df["file_name"].str.startswith("GBP")
]
avg_daily_volume_df = avg_daily_volume_df.loc[
    ~avg_daily_volume_df["file_name"].str.startswith("EUR")
]
avg_daily_volume_df

In [ ]:
avg_daily_volume_df.head(100)

In [ ]:
sample_file_names = avg_daily_volume_df.head(100)["file_name"].to_list()
sample_file_names

In [ ]:
df = get_market_data(sample_file_names[0], start_time=start_time, end_time=end_time)
df

In [ ]:
N = 1
currency_file_pairs = list(itertools.combinations(sample_file_names, 2))
currency_file_pairs = random.sample(currency_file_pairs, N)
print(f"Number of currency pair(s): {len(currency_file_pairs)}")
display(currency_file_pairs)

In [ ]:
close_1_df = get_market_data(
    currency_file_pairs[0][0], start_time=start_time, end_time=end_time
)
close_1_df

In [ ]:
close_2_df = get_market_data(
    currency_file_pairs[0][1],
    start_time=start_time - dt.timedelta(days=30),
    end_time=end_time,
)
close_2_df

In [ ]:
pairs_trading_df = pd.DataFrame(
    index=pd.date_range(start=start_time, end=end_time, freq=dt.timedelta(minutes=1))
)
pairs_trading_df = pd.merge_asof(
    pairs_trading_df,
    close_1_df,
    right_index=True,
    left_index=True,
    suffixes=("_1", "_2"),
)
pairs_trading_df = pd.merge_asof(
    pairs_trading_df,
    close_2_df,
    right_index=True,
    left_index=True,
    suffixes=("_1", "_2"),
)
pairs_trading_df

In [ ]:
cointegration_result = stochastic.RollingCointegration(
    pairs_trading_df["close_1"], pairs_trading_df["close_2"], window=7 * 24 * 60
).fit()
cointegration_result

In [ ]:
pairs_trading_df["beta"] = cointegration_result.beta
pairs_trading_df["pvalue"] = cointegration_result.pvalue

In [ ]:
ou_result = stochastic.RollingOrnsteinUhlenbeck(
    pairs_trading_df["beta"].to_numpy(),
    pairs_trading_df["close_1"].to_numpy(),
    pairs_trading_df["close_2"].to_numpy(),
    window=7 * 24 * 60,
).fit()
ou_result

In [ ]:
pairs_trading_df["mu"] = ou_result.mu
pairs_trading_df["sigma"] = ou_result.sigma
pairs_trading_df["theta"] = ou_result.theta
pairs_trading_df["half_life"] = ou_result.theta

In [ ]:
pairs_trading_df.loc[
    pairs_trading_df["pvalue"] > 0.001, ["mu", "sigma", "theta", "half_life"]
] = np.nan

In [ ]:
pairs_trading_df

In [ ]:
stochastic.OrnsteinUhlenbeck.get_optimal_exit_level

In [ ]:
to_asset_name = "DOT"
from_asset_name = "USD"

path = (
    f"/Users/glynfinck/Downloads/Kraken_OHLCVT/{to_asset_name}{from_asset_name}_1.csv"
)

# Read CSV without headers
df = pd.read_csv(path, header=None)

# Set meaningful column names for OHLCV data
df.columns = ["timestamp", "open", "high", "low", "close", "volume", "trade_count"]

# Format the timestamp to be a datetime object
df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s")
df["from_asset_name"] = from_asset_name
df["to_asset_name"] = to_asset_name
df["open"] = df["open"].astype(float)
df["high"] = df["high"].astype(float)
df["low"] = df["low"].astype(float)
df["close"] = df["close"].astype(float)
df["volume"] = df["volume"].astype(float)
df["trade_count"] = df["trade_count"].astype(int)

df.drop(columns=["trade_count"], inplace=True)

df = df.sort_values("timestamp")

df = df.loc[df["timestamp"] >= start_time].reset_index(drop=True)

# Display the first few rows of the dataframe
df

In [ ]:
gbm_params_1 = stochastic.GeometricBrownianMotionResult(mu=0.00001, sigma=0.001)
gbm_1 = stochastic.GeometricBrownianMotion(gbm_params_1)
ou_params = stochastic.OrnsteinUhlenbeckResult(mu=0.0005, sigma=0.001, theta=0.0001)
ou = stochastic.OrnsteinUhlenbeck(ou_params)
beta = 0.5
timestamps = pd.date_range(start=start_time, end=end_time, freq=dt.timedelta(minutes=1))
N = len(timestamps)
close_1 = gbm_1.simulate(N, 1, 100).flatten()
close_2 = beta * close_1 + ou.simulate(N, 1, 0.01).flatten()
df = pd.DataFrame(
    {
        "timestamp": timestamps,  # Already a Series/array
        "close_1": close_1,  # Already flattened numpy array
        "close_2": close_2,  # Already flattened numpy array
    }
)